# **G345 ANÁLISIS DE DATOS**

# ANÁLISIS DE TARIFAS Y COSTOS DE ENERGÍA ELÉCTRICA PARA EL MERCADO REGULADO

# Nombre y cédula de los integrantes

- Ximena Orbes Arias - 1033099573

- Juan Esteban Lame Vasquez - 1142514863

- Isabella Cardona Betancourt -

- Meycol Kleym Quintero Castaño -

# Preparar el entorno

In [ ]:
# (Opcional) Instalar dependencias en este notebook
# Si ya están instaladas, esta celda terminará rápido.
# %pip -q install pandas numpy matplotlib

In [ ]:
# Se importan las librerías
import pandas as pd
import numpy as np
import os
from pathlib import Path

# Base de datos: Tarifas y Costos de Energía Eléctrica para el Mercado Regulado

## Acerca de este archivo

En cumplimiento de las Leyes 142 de 1994 y 1955 de 2019, y las Resoluciones CREG (Comisión de Regulación de Energía y Gas) 058 de 2000, 119 de 2007, 105 de 2009, 156 de 2009, 173 de 2011, 152 de 2018 y 101028 de 2023, este conjunto de datos proporciona información detallada sobre las tarifas y costos de energía eléctrica en el mercado regulado.

Los datos incluyen información sobre las tarifas de energía, el costo de compra de energía, los cargos de transporte en los sistemas de transmisión y distribución, los márgenes de comercialización, costos relacionados con generación y pérdidas, así como otros factores operativos que influyen en el precio final que los usuarios deben pagar. Además, se desglosan por tipo de propiedad (Operador de Red, Propiedad Compartida, Propiedad Cliente) y se dividen en diferentes niveles de tensión eléctrica.

Este conjunto de datos está diseñado para facilitar el análisis de las tarifas y costos eléctricos en distintas regiones y periodos, proporcionando una visión clara de los componentes que afectan el precio de la energía para los usuarios del mercado regulado.

Los datos abarcan el periodo de enero de 2024 a septiembre de 2025.

## Diccionario de datos

- *Año:* El año en el que se registran los datos sobre tarifas y costos de energía eléctrica. Ejemplo: 2024
- *Priodo:* El mes o período específico dentro del año al que corresponde la tarifa o costo de energía.
- *Operador de red:* La empresa responsable de la distribución de energía eléctrica en una zona específica.
- *Nivel:* La categoría del servicio eléctrico, que puede incluir diferentes tipos de tensión como Baja Tensión (B.T.), Media Tensión (M.T.) o Alta Tensión (A.T.).
- *CU Total:* El costo total por kilovatio hora (kWh) que los usuarios deben pagar por la electricidad, incluyendo todos los cargos.
- *Costo Compra (Gm,i):* El precio que paga el operador de red por la electricidad comprada a los generadores.
- *Cargo Transporte STN (Tm):* El costo asociado al transporte de electricidad a través del Sistema de Transmisión Nacional (STN), que es la red de alta tensión que lleva la electricidad de los generadores a las subestaciones.
- *Cargo Transporte SDL (Dn,m):* El costo asociado al transporte de electricidad en el Sistema de Distribución Local (SDL), que distribuye la electricidad desde las subestaciones hasta los usuarios finales.
- *Margen Comercialización (CVm,i,j):* El costo adicional que cubre la comercialización de la electricidad, incluyendo los gastos de operación y transacción del operador de red.
- *Costo G, T, Pérdidas (PRn,m):* Los costos asociados a la generación, transmisión y pérdidas de energía durante su transporte por la red, debido a la resistencia y otros factores.

## Archivo CSV

In [ ]:
# Ruta relativa: este notebook está en la misma carpeta que DATASET.csv
ruta_csv = Path("Tarifas_y_Costos_de_Energía_Eléctrica_para_el_Mercado_Regulado.csv")

try:
    if not ruta_csv.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta_csv.resolve()}")

    df = pd.read_csv(ruta_csv, low_memory=False)
except (OSError, FileNotFoundError) as err:
    print(f"No se pudo abrir el archivo. Error: {err}.")
else:
    print("Archivo cargado con éxito en el dataframe.")
    print("Dimensiones:", df.shape)
    display(df.head())

# Limpieza y Preparación de Datos

## Información sobre el CSV

In [ ]:
# Observamos los tipos de datos que se tienen en la base de datos para empezar a visualizar y tener en cuenta
# qué columnas modificar

df.info()

In [ ]:
# Si necesitamos llamar una función en una columna que requiera el nombre de esta, podemos venir a este Index de
# columnas para facilitar la busqueda del nombre, este también sería una versión más simple y enfocada a lo necesario de df.info()

for i, col in enumerate(df.columns):
    nulos_con_null = df[col].isnull().sum()
    nulos_con_nan = df[col].isna().sum()
    print(f"""Columna {i}: '{col}' → {df[col].dtype} | Nulos: {nulos_con_null} (null), {nulos_con_nan} (nan)  
Cantidad de datos únicos: {df[col].nunique()}
Duplicados: {df[col].duplicated().sum()}
""")

In [ ]:
# Por si hay necesidad de ver los diferentes datos que hay en una columna
# print(df['nombre_columna'].unique())

## Modificación de tipo de datos

In [ ]:
# Para facilitar la conversión de 'object' a flotante: creamos una función que me identifique las comas y las pase a puntos

def conversionFloat(df, col, type):
    df[col] = (df[col]
               .astype(str)
               .str.replace(r'[^\d.]', '', regex=True)
               .astype(type)
                        )

In [ ]:
# Conversión de datos para su uso en el análisis

conversionFloat(df, 'CU Total', float)
conversionFloat(df, 'Cfm,j ($/fact.)', float)
conversionFloat(df, 'Año', int)
print(f"""Primeros 5 datos de CU Total
{df['CU Total'].head()}

Primeros 5 datos de Cfm,j ($/fact.) 
{df['Cfm,j ($/fact.)'].head()}

Primeros 5 datos de Año 
{df['Año'].head()}""")

In [ ]:
# Limpieza de fechas

df['Num_Periodo'] = (df['Periodo']
                       .str.replace('Enero', '1')
                       .str.replace('Febrero', '2')
                       .str.replace('Marzo', '3')
                       .str.replace('Abril', '4')
                       .str.replace('Mayo', '5')
                       .str.replace('Junio', '6')
                       .str.replace('Julio', '7')
                       .str.replace('Agosto', '8')
                       .str.replace('Septiembre', '9')
                       .str.replace('Octubre', '10')
                       .str.replace('Noviembre', '11')
                       .str.replace('Diciembre', '12')
                       )

df.head()

In [ ]:
# Pasarlo a formato fecha

df['Fecha'] = pd.to_datetime(
    dict(year=df['Año'], month=df['Num_Periodo'], day=1)
).dt.to_period('M')

df.head()

In [ ]:
# Corroboración entre nombres de operadores en la columna 'Operador de red'

print(df['Operador de red'].unique())

df[df['Operador de red'].isin(['CELSIA Colombia - Tolima', 'CELSIA - Tolima'])].head(10)


In [ ]:
# Cambio de nombre en empresas que tienen cambio de nombre pero son las mismas

df['Operador de red'] = df['Operador de red'].replace({'CELSIA Colombia - Tolima': 'CELSIA - Tolima'})
df['Operador de red'] = df['Operador de red'].replace({'CELSIA Colombia - Valle del Cauca': 'CELSIA - Valle del Cauca'})
df['Nivel'] = df['Nivel'].replace({'Nivel 1 ( Propiedad OR )': 'Nivel 1 (Propiedad OR)'})
df['Nivel'] = df['Nivel'].replace({'Nivel 1  (Propiedad Cliente)': 'Nivel 1 (Propiedad OR)'})


print(df['Operador de red'].unique())
print(df['Nivel'].unique())

### Categorización

In [ ]:
# Categorizamos las empresas en Operador de red y Nivel

df['OPR'] = df['Operador de red'].astype('category').cat.codes
df['OPR'].unique()

# Definir el orden en la columna 'Nivel'

categorias_ordenadas = ['Nivel 1 (Propiedad OR)', 'Nivel 1 (Propiedad Compartida)',
 'Nivel 1 (Propiedad Cliente)', 'NIVEL II', 'NIVEL III']

# Crear la serie categórica ordenada
serie_categorica = pd.Categorical(df['Nivel'], categories=categorias_ordenadas, ordered=True)

# Obtener los códigos numéricos
df['NIV'] = serie_categorica.codes

# Verificar el mapeo y los resultados
print("Mapeo de categorías a códigos:")
for codigo, categoria in enumerate(serie_categorica.categories):
    print(f"{categoria} → {codigo}")

serie_categorica

# Análisis de datos

## Preguntas a responder con la base de datos

## ANÁLISIS